# 02 - Results and error analysis
Run after `scripts/01`, `02` and `03`. Looks at **where** the chosen model fails, which is more useful than another metric.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # run the notebook from /notebooks
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import average_precision_score

from src import config
from src.features import load_data, add_features, time_split, NUM, CAT

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

import json

## 1. Leaderboard

In [ ]:
rep = pd.read_csv(config.OUT / "final_report.csv")
best = json.load(open(config.OUT / "best_model.json"))
print(best)
rep.round(3)

## 2. Ensemble weights
Which base models do the blends actually rely on?

In [ ]:
w = json.load(open(config.OUT / "ensemble_weights.json"))
pd.DataFrame(w).round(3)

## 3. Score distribution of the chosen model on the test period

In [ ]:
tst = pd.read_csv(config.OUT / "test_preds.csv").merge(pd.read_csv(config.OUT / "test_ens.csv").drop(columns=["y", "amount"]), on="transaction_id")
test = time_split(add_features(load_data()))[1].merge(tst[["transaction_id", best["best_model"]]].rename(columns={best["best_model"]: "score"}), on="transaction_id")
test["alert"] = (test.score >= best["threshold"]).astype(int)
fig, ax = plt.subplots(figsize=(8, 3.5))
for label, g in test.groupby(config.TARGET):
    sns.histplot(g.score, bins=50, ax=ax, label=f"fraud={label}", stat="density", common_norm=False, alpha=.5)
ax.axvline(best["threshold"], c="k", ls="--", label="threshold"); ax.set_yscale("log"); ax.legend()

## 4. Missed fraud (false negatives) - what do they look like?

In [ ]:
fn = test[(test.alert == 0) & (test.is_fraud == 1)]
tp = test[(test.alert == 1) & (test.is_fraud == 1)]
print(f"caught {len(tp)} | missed {len(fn)} | missed amount {fn.amount.sum():,.0f} of {test[test.is_fraud==1].amount.sum():,.0f}")
cmp = pd.DataFrame({"missed_median": fn[NUM].median(), "caught_median": tp[NUM].median()}).round(2)
cmp

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["merchant_category", "channel", "country"]):
    pd.DataFrame({"missed": fn[col].value_counts(normalize=True), "caught": tp[col].value_counts(normalize=True)}).fillna(0).plot(kind="bar", ax=ax, title=f"{col}: share of frauds")
plt.tight_layout()

## 5. False alarms - which segments burn analyst time?

In [ ]:
fp = test[(test.alert == 1) & (test.is_fraud == 0)]
seg = test.groupby("merchant_category").agg(n=("alert", "size"), alerts=("alert", "sum"), frauds=("is_fraud", "sum"))
seg["precision"] = (test[test.alert == 1].groupby("merchant_category").is_fraud.mean()).round(3)
seg["alert_rate"] = (seg.alerts / seg.n).round(3)
seg.sort_values("alerts", ascending=False)

## 6. Performance by month (stability)

In [ ]:
test["month"] = test.timestamp.dt.to_period("M")
test.groupby("month").apply(lambda g: pd.Series({
    "n": len(g), "fraud_rate": g.is_fraud.mean(),
    "pr_auc": average_precision_score(g.is_fraud, g.score) if g.is_fraud.sum() else np.nan,
    "recall": g[g.is_fraud == 1].alert.mean()})).round(3)

## Takeaways (fill in)
- Which fraud patterns are missed and what feature could catch them
- Segments where the threshold should differ
- Is performance stable month to month?